# Extract entities and relationships from storm reports

Use a pretrained text model to find places and relationships in ten NOAA flash-flood narratives. Run this sample in a signed-in ArcGIS Pro environment with the deep learning libraries required by the model and a writable workspace. The results are text tables, not geocoded locations.

In [1]:
import os

import arcpy
import pandas as pd
from IPython.display import display

from arcgis.gis import GIS

## 1. Set the workspace

Choose one folder to save the model and outputs.

In [2]:
workspace = os.path.join(os.getcwd(), "named_entity_relationship_extraction")
os.makedirs(workspace, exist_ok=True)
gdb = os.path.join(workspace, "results.gdb")
if not arcpy.Exists(str(gdb)):
    arcpy.management.CreateFileGDB(workspace, os.path.basename(gdb))
arcpy.env.overwriteOutput = True

## 2. Load the model and reports

Connect with your signed-in account and download the Living Atlas model into the workspace.
The DLPK size is approximately 1.7 GB.

In [ ]:
gis = GIS("pro")
model_item = gis.content.get("e0eeb8abf73441baa455f40a744fec93")
MODEL = model_item.download(save_path=str(workspace))
print(model_item.title)

Open the NOAA Storm Events item and select its first layer.
This is the source of the report text used in the sample.

In [ ]:
item = gis.content.get("88cc0d5e55f343c28739af1a091dfc91")
layer = item.layers[0]
print(item.title)

Select 10 nonempty flash-flood reports from 2010, ordered by event ID.

In [ ]:
where = "YEAR = 2010 AND EVENT_TYPE = 'Flash Flood' AND EVENT_NARRATIVE IS NOT NULL AND EVENT_NARRATIVE <> ''"
records = layer.query(
    where=where,
    out_fields="EVENT_ID,EVENT_NARRATIVE",
    order_by_fields="EVENT_ID",
    return_geometry=False,
    result_record_count=10,
    return_all_records=False,
)

Keep the event IDs and narratives in a DataFrame and preview three reports.

In [ ]:
reports = records.sdf[["EVENT_ID", "EVENT_NARRATIVE"]].copy()
reports["EVENT_ID"] = reports["EVENT_ID"].astype(str)
with pd.option_context("display.max_colwidth", 160):
    display(reports.head(3))

,EVENT_ID,EVENT_NARRATIVE
0,203080,Sheriff reported flooding at apartment complex with up to a foot of water in several apartments.
1,203081,"Sheriff reported flooding along several streams into businesses and homes, especially along Rocky Creek."
2,203089,Sheriff reported flooding in Graniteville along Canal Street.


## 3. Prepare the input table

Create the input directly in the workspace geodatabase, with one field for the ID and one for the text.

In [ ]:
input_table = os.path.join(gdb, "storm_reports")
arcpy.management.CreateTable(str(gdb), "storm_reports")
arcpy.management.AddField(input_table, "EVENT_ID", "TEXT", field_length=30)
text_length = int(reports["EVENT_NARRATIVE"].str.len().max())
arcpy.management.AddField(
    input_table, "EVENT_NARRATIVE", "TEXT", field_length=text_length
)

Insert the reports into the saved input table.
The model will process each narrative as a separate record.

In [ ]:
with arcpy.da.InsertCursor(input_table, ["EVENT_ID", "EVENT_NARRATIVE"]) as cursor:
    for row in reports.itertuples(index=False, name=None):
        cursor.insertRow(row)

## 4. Extract entities

Choose the entity labels and a threshold of 0.5.

In [ ]:
entity_classes = "location,road,river"
threshold = "0.5"
inputs = dict(
    in_layer=input_table,
    data_fields=["EVENT_NARRATIVE"],
    in_model_definition_file=str(MODEL),
)

Set the task to NER and choose the output table inside the workspace geodatabase.
A batch size of one keeps memory use low for this small sample.

In [ ]:
entity_table = os.path.join(gdb, "entities")
ner_arguments = [
    ["task_type", "NER"],
    ["entity_classes", entity_classes],
    ["threshold", threshold],
    ["batch_size", "1"],
]

Run entity extraction.

In [ ]:
ner_run = arcpy.geoai.ProcessTextUsingAIModel(
    **inputs,
    out_layer=entity_table,
    model_arguments=ner_arguments,
)
arcpy.management.JoinField(
    entity_table, "Input_Text", input_table, "EVENT_NARRATIVE", ["EVENT_ID"]
)
print(ner_run.getMessages())

Read the saved table into pandas to display the extracted entities and confidence scores.
NaN means no match passed the threshold; multiple matches appear as comma-separated text.

In [ ]:
fields = [field.name for field in arcpy.ListFields(entity_table) if field.type != "OID"]
with arcpy.da.SearchCursor(entity_table, fields) as cursor:
    entities = pd.DataFrame(list(cursor), columns=fields)
entities

,Input_Text,location,road,river,location_confidence_score,road_confidence_score,river_confidence_score,EVENT_ID
0,Sheriff reported flooding at apartment complex...,"apartment complex, apartments",NaN,NaN,"0.9915, 0.8517",NaN,NaN,203080
1,Sheriff reported flooding along several stream...,NaN,NaN,Rocky Creek,NaN,NaN,0.9951,203081
2,Sheriff reported flooding in Graniteville alon...,Graniteville,Canal Street,flooding,0.9951,0.9825,0.7751,203089
3,Flooding was reported across numerous roadways...,"Emerson, Arkansas",numerous roadways,Flooding,"0.9206, 0.8985",0.9272,0.7576,203452
4,Heavy rain caused rapid rises and flash floodi...,"towns, Winfield, Lewisburg",roads,"small streams, creeks","0.5433, 0.9949, 0.9955",0.977,"0.9465, 0.979",204337
5,Heavy rain during the morning hours caused sig...,Benton,"Fishing Creek, dyke system",Little Fishing Creek,0.8844,"0.9268, 0.6579",0.9564,204340
6,A car was apparently swept off the road during...,NaN,road,Cibolo Creek,NaN,0.9773,0.9614,205181
7,Thunderstorms produced around 3 inches of rain...,banks,roads,streams,0.9252,0.9955,0.9894,205474
8,About 18 inches of water flooded roads and hom...,Arizona City,"roads, Toki","water, Blackwater",0.9836,"0.952, 0.8992","0.7551, 0.9016",205556
9,A spotter reported that a driveway was washed ...,driveway,streets,water,0.8856,0.9744,0.8223,206922


## 5. Extract relationships

Choose located in and located near to find containment and proximity relationships.
Write these predictions to a separate table in the same geodatabase.

In [ ]:
relationship_table = os.path.join(gdb, "relationships")
relationship_classes = "located in,located near"

Switch the task to Relationship Extraction and supply the relationship labels.
Keep the same entity labels, threshold, and batch size used above.

In [ ]:
relation_arguments = [
    ["task_type", "Relationship Extraction"],
    ["entity_classes", entity_classes],
    ["relationship_classes", relationship_classes],
    ["threshold", threshold],
    ["batch_size", "1"],
]

Run relationship extraction directly into its saved table and attach event IDs.
One report can produce several relationships, so row-count warning 260340 can be expected.

In [ ]:
relationship_run = arcpy.geoai.ProcessTextUsingAIModel(
    **inputs,
    out_layer=relationship_table,
    model_arguments=relation_arguments,
)
arcpy.management.JoinField(
    relationship_table, "Input_Text", input_table, "EVENT_NARRATIVE", ["EVENT_ID"]
)
print(relationship_run.getMessages())

Display the saved relationship table and compare predictions with the original input text.
Read each row as Head_Text -- Relation --> Tail_Text; null relationship fields indicate no detection.

In [ ]:
fields = [
    field.name for field in arcpy.ListFields(relationship_table) if field.type != "OID"
]
with arcpy.da.SearchCursor(relationship_table, fields) as cursor:
    relationships = pd.DataFrame(list(cursor), columns=fields)
relationships

,Input_Text,Head_Text,Tail_Text,Relation,Confidence_Score,EVENT_ID
0,Sheriff reported flooding at apartment complex...,apartment complex,apartments,located in,0.950194,203080
1,Sheriff reported flooding along several stream...,streams,Rocky Creek,located near,0.962077,203081
2,Sheriff reported flooding along several stream...,businesses,Rocky Creek,located near,0.974594,203081
3,Sheriff reported flooding along several stream...,homes,Rocky Creek,located near,0.976506,203081
4,Sheriff reported flooding in Graniteville alon...,flooding,Graniteville,located in,0.990415,203089
5,Sheriff reported flooding in Graniteville alon...,flooding,Canal Street,located near,0.929908,203089
6,Sheriff reported flooding in Graniteville alon...,Canal Street,Graniteville,located in,0.540193,203089
7,Flooding was reported across numerous roadways...,roadways,Emerson,located near,0.985733,203452
8,Flooding was reported across numerous roadways...,Emerson,Arkansas,located in,0.979177,203452
9,Heavy rain caused rapid rises and flash floodi...,small streams,creeks,located near,0.571209,204337


## Conclusion

This notebook demonstrated how to use the Named Entity and Relationship Extraction pretrained model to identify locations, roads, rivers, and relationships in NOAA storm reports. This workflow can be used as a starting point for organizing descriptive storm information and supporting further analysis.

## References

- NOAA/NCEI, Esri: [NOAA USA Storm Events Database](https://www.arcgis.com/home/item.html?id=88cc0d5e55f343c28739af1a091dfc91).
- Esri pretrained model guide: [Use the Named Entity and Relationship Extraction model](https://doc.arcgis.com/en/pretrained-models/latest/text/using-named-entity-and-relationship-extraction.htm).